In [1]:
import os
import sys
import numpy as np
import torch

# add parent directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import *
from src.data_loader import load_data
from src.preprocess import preprocess_list
from src.embedder import load_phobert, embed_texts
from src.trainer import train_svm, evaluate
from src.cso_optimizer import optimize_cso
from sklearn.model_selection import train_test_split
from src.embedding_cache import get_or_create_embeddings
from src.split_cache import get_or_create_split

c:\Users\nguye\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
texts, labels = load_data(DATA_PATH)

print("Samples:", len(texts))

Columns: ['title', 'label']
Loaded samples: 585
Samples: 585


In [3]:
texts = preprocess_list(texts)

In [21]:
tokenizer, model = load_phobert(PHOBERT_DIR)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1735.06it/s]


In [5]:
# X = embed_texts(texts, tokenizer, model)
# y = labels

EMBEDDING_DIR = "models/cache_embeddings"

X, y = get_or_create_embeddings(
    texts=texts,
    labels=labels,
    embed_fn=embed_texts,
    tokenizer=tokenizer,
    model=model,
    save_dir=EMBEDDING_DIR
)

✅ Loaded cached embeddings:
   X: (585, 768)
   y: (585,)


In [6]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=TEST_SIZE,
#     stratify=y,
#     random_state=RANDOM_STATE
# )

SPLIT_DIR = "models/cache_split"

X_train, X_test, y_train, y_test = get_or_create_split(
    X, y,
    save_dir=SPLIT_DIR
)

✅ Loaded cached split


In [7]:
clf = train_svm(X_train, y_train)
evaluate(clf, X_test, y_test)

Accuracy: 0.5128205128205128
F1: 0.49644374522891144
              precision    recall  f1-score   support

           0       0.82      1.00      0.90         9
           1       0.43      0.33      0.38         9
           2       0.50      0.33      0.40         9
           3       0.57      0.44      0.50         9
           4       0.62      0.89      0.73         9
           5       0.36      0.56      0.43         9
           6       0.58      0.78      0.67         9
           7       0.75      0.67      0.71         9
           8       0.50      0.44      0.47         9
           9       0.38      0.33      0.35         9
          10       0.20      0.11      0.14         9
          11       0.67      0.67      0.67         9
          12       0.11      0.11      0.11         9

    accuracy                           0.51       117
   macro avg       0.50      0.51      0.50       117
weighted avg       0.50      0.51      0.50       117



In [8]:
best_C = optimize_cso(X_train, y_train)
print("Best C:", best_C)

C=383.2622 -> 0.5566
C=6607.6920 -> 0.5592
C=5.6627 -> 0.5855
C=572.7978 -> 0.5592
C=3.4123 -> 0.6024
C=15.9475 -> 0.5687
C=6575.5720 -> 0.5592
C=418.0085 -> 0.5566
C=5.6460 -> 0.5855
C=585.9593 -> 0.5592
C=15.8492 -> 0.5687
C=4976.3662 -> 0.5592
C=150.5951 -> 0.5566
Iter 0 | Best fitness: -0.6024
C=6617.8070 -> 0.5592
C=377.8287 -> 0.5566
C=5.5191 -> 0.5855
C=610.8686 -> 0.5592
C=15.5748 -> 0.5707
C=1.0790 -> 0.6099
C=0.6156 -> 0.6117
Iter 1 | Best fitness: -0.6117
C=6163.3959 -> 0.5592
C=414.6915 -> 0.5566
C=1.0845 -> 0.6099
C=558.9092 -> 0.5592
C=14.7638 -> 0.5684
C=3.6198 -> 0.6007
C=0.0155 -> 0.5371
Iter 2 | Best fitness: -0.6117

Best solution: [-0.21070745]
Best fitness: -0.611739584143728
Best C: 0.6155914134922305


In [9]:
best_model = train_svm(X_train, y_train, C=best_C)
evaluate(best_model, X_test, y_test)

Accuracy: 0.5128205128205128
F1: 0.495878134369183
              precision    recall  f1-score   support

           0       0.82      1.00      0.90         9
           1       0.43      0.33      0.38         9
           2       0.50      0.33      0.40         9
           3       0.50      0.44      0.47         9
           4       0.62      0.89      0.73         9
           5       0.36      0.56      0.43         9
           6       0.58      0.78      0.67         9
           7       0.75      0.67      0.71         9
           8       0.50      0.44      0.47         9
           9       0.43      0.33      0.38         9
          10       0.20      0.11      0.14         9
          11       0.67      0.67      0.67         9
          12       0.11      0.11      0.11         9

    accuracy                           0.51       117
   macro avg       0.50      0.51      0.50       117
weighted avg       0.50      0.51      0.50       117



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

# Linear
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=5000
    ))
])

# RBF
# rbf_model = Pipeline([
#     ("scaler", StandardScaler()),
#     ("svm", SVC(
#         kernel="rbf",
#         C=10,
#         gamma="scale",
#         class_weight="balanced"
#     ))
# ])

print(">> Training LinearSVC...")
linear_model.fit(X_train, y_train)

# print(">> Training RBF SVC...")
# rbf_model.fit(X_train, y_train)

>> Training LinearSVC...
>> Training RBF SVC...


,steps,"[('scaler', ...), ('svm', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,10
,kernel,'rbf'
,degree,3
,gamma,'scale'


In [ ]:
# from src.ensemble import ensemble_predict

# y_pred_ensemble = ensemble_predict(
#     [linear_model, rbf_model],
#     X_test
# )

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# acc = accuracy_score(y_test, y_pred_ensemble)
# f1 = f1_score(y_test, y_pred_ensemble, average="macro")

# print(f"Ensemble Accuracy : {acc:.4f}")
# print(f"Ensemble F1-macro : {f1:.4f}")
# print("\nClassification report:")
# print(classification_report(y_test, y_pred_ensemble))

y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy : {acc:.4f}")
print(f"F1-macro : {f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Ensemble Accuracy : 0.5470
Ensemble F1-macro : 0.5246

Classification report:
              precision    recall  f1-score   support

           0       0.75      1.00      0.86         9
           1       0.40      0.44      0.42         9
           2       0.57      0.44      0.50         9
           3       0.42      0.56      0.48         9
           4       0.54      0.78      0.64         9
           5       0.42      0.56      0.48         9
           6       0.64      0.78      0.70         9
           7       0.80      0.89      0.84         9
           8       0.57      0.44      0.50         9
           9       0.38      0.33      0.35         9
          10       0.25      0.11      0.15         9
          11       0.86      0.67      0.75         9
          12       0.25      0.11      0.15         9

    accuracy                           0.55       117
   macro avg       0.53      0.55      0.52       117
weighted avg       0.53      0.55      0.52       117



In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import accuracy_score, f1_score

results = {}

In [ ]:
# linear_model = Pipeline([
#     ("scaler", StandardScaler()),
#     ("svm", LinearSVC(
#         C=1.0,
#         class_weight="balanced",
#         max_iter=5000
#     ))
# ])

# linear_model.fit(X_train, y_train)
# y_pred = linear_model.predict(X_test)

# results["LinearSVC"] = (
#     accuracy_score(y_test, y_pred),
#     f1_score(y_test, y_pred, average="macro")
# )

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=best_C,
        class_weight="balanced",
        max_iter=5000
    ))
])

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"F1-macro: {f1:.4f}")

In [15]:
linear_cso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=best_C,
        class_weight="balanced",
        max_iter=5000
    ))
])

linear_cso_model.fit(X_train, y_train)
y_pred = linear_cso_model.predict(X_test)

results["LinearSVC + CSO"] = (
    accuracy_score(y_test, y_pred),
    f1_score(y_test, y_pred, average="macro")
)

In [16]:
rbf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced"
    ))
])

rbf_model.fit(X_train, y_train)
y_pred = rbf_model.predict(X_test)

results["SVC RBF"] = (
    accuracy_score(y_test, y_pred),
    f1_score(y_test, y_pred, average="macro")
)

In [17]:
for name, (acc, f1) in results.items():
    print(f"{name}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-macro: {f1:.4f}")
    print("-" * 30)

LinearSVC
  Accuracy: 0.5128
  F1-macro: 0.4964
------------------------------
LinearSVC + CSO
  Accuracy: 0.5128
  F1-macro: 0.4959
------------------------------
SVC RBF
  Accuracy: 0.5897
  F1-macro: 0.5918
------------------------------


In [18]:
import joblib
import os

os.makedirs("models", exist_ok=True)

best_model = rbf_model

joblib.dump(best_model, "models/final_model.joblib")

print("Saved final model (SVC RBF)")

Saved final model (SVC RBF)


In [25]:
import json

with open("../src/label_map.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)

label_map = {int(k): v for k, v in label_map.items()}

In [28]:
def predict_text(text, model, tokenizer, phobert):
    from src.preprocess import preprocess_text
    from src.embedder import embed_texts

    text = preprocess_text(text)
    emb = embed_texts([text], tokenizer, phobert)

    pred = model.predict(emb)[0]
    # return pred
    return label_map[int(pred)]

In [35]:
from src.embedder import load_phobert

_, phobert = load_phobert(PHOBERT_DIR)

model = joblib.load("models/final_model.joblib")

predict_text(
    "Việt Nam phát hiện nhiều ca nhiễm biến thể Omicron mới",
    model,
    tokenizer,
    phobert
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1254.91it/s]


'sức khỏe'